# Vọng: chạy website TTS trên Kaggle

Notebook này dựng website (FastAPI + giao diện) trên GPU của Kaggle và mở một đường link công khai bằng Cloudflare Tunnel.

**Trước khi chạy:** Settings, Accelerator: **GPU T4 x2** (hoặc P100), **Internet: On**.
Tải thư mục dự án (`backend/`, `frontend/`, `requirements.txt`) lên làm một Kaggle Dataset, ví dụ tên `vong-tts`, rồi Add Data vào notebook.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Chép mã nguồn và cài thư viện

In [ ]:
import os, shutil, glob

# Đổi đường dẫn nếu dataset của bạn tên khác. Có thể thay bằng: !git clone <repo-của-bạn> /kaggle/working/vong
hits = glob.glob("/kaggle/input/**/backend/app.py", recursive=True)
assert hits, "Không thấy backend/app.py trong /kaggle/input. Đã Add Data chưa?"
SRC = os.path.dirname(os.path.dirname(hits[0]))
APP = "/kaggle/working/vong"
if os.path.exists(APP):
    shutil.rmtree(APP)
shutil.copytree(os.path.realpath(SRC), APP, ignore=shutil.ignore_patterns("cache", ".venv", ".hallmark"))
print(os.listdir(APP))

In [ ]:
!pip install -q omnivoice fastapi "uvicorn[standard]" python-multipart

## 2. Khởi động máy chủ

Model được nạp một lần khi server khởi động (lần đầu tải trọng số từ Hugging Face mất vài phút). Cache nằm ở `/kaggle/working/vong-cache`, nên voice prompt và các câu đã sinh vẫn còn nếu bạn khởi động lại server trong cùng phiên.

In [ ]:
import subprocess, time, json, urllib.request

env = dict(os.environ, VONG_MOCK="0", VONG_CACHE_DIR="/kaggle/working/vong-cache", PYTHONUNBUFFERED="1")
log = open("/kaggle/working/server.log", "w")
server = subprocess.Popen(["python", "-m", "backend.serve", "--port", "8000"], cwd=APP, env=env,
                          stdout=log, stderr=subprocess.STDOUT)

state = None
for _ in range(600):
    time.sleep(2)
    try:
        s = json.load(urllib.request.urlopen("http://127.0.0.1:8000/api/status"))
        state = s["engines"]["omnivoice"]
        if state["state"] in ("ready", "error"):
            break
    except Exception:
        pass
print(state)
if not state or state["state"] != "ready":
    print(open("/kaggle/working/server.log").read()[-3000:])

## 3. Mở link công khai (Cloudflare quick tunnel, không cần tài khoản)

In [ ]:
import re
if not os.path.exists("/kaggle/working/cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "/kaggle/working/cloudflared"], check=True)
    os.chmod("/kaggle/working/cloudflared", 0o755)

tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
print("Mở website tại:", url)

## 4. Khi cần

- Xem log: `!tail -n 50 /kaggle/working/server.log`
- Dừng: `server.terminate(); tunnel.terminate()`
- GPU hết bộ nhớ: giảm “Số câu mỗi lô” trong phần Xử lý văn bản và hiệu năng.

In [ ]:
!tail -n 30 /kaggle/working/server.log